# Домашнее задание 2: Сервис курсов валют

Вы пишете внутренний сервис банка для конвертации валют - чтобы во всём приложении была одна точка истины (фронту, мобилке и бэкенду одинаково отвечать сколько EUR за USD). Источник - открытый API `api.frankfurter.dev` (исторические курсы Европейского центрального банка).

Сервис собирается из 6 компонентов в таком порядке:

1. `fetch_rates` - тянем сырые курсы из API.
2. `Currency` - модель одной валюты.
3. `RateBook` - коллекция валют с быстрым поиском по коду.
4. `Converter` - умеет конвертировать (прямо и через кросс-курс).
5. `CachedRateBook` - наследник `RateBook` с логированием обращений.
6. `TurboBook` - то же + миксин для подписки на события (нужно для определения MRO).

После реализации запустите финальную ячейку и впишите выводы Q1-Q6 в инпуты на странице задания. Q7 - открытый вопрос.

## Источник

URL: `https://api.frankfurter.dev/v1/2024-01-15?from=USD`

Дата зафиксирована намеренно (исторический эндпоинт) - так курсы не меняются между запусками, и автоматическая проверка работает стабильно. В реальном продакшене URL был бы без даты (`/latest`).

Ответ - JSON вида:
```json
{
  "amount": 1.0,
  "base": "USD",
  "date": "2024-01-15",
  "rates": {
    "EUR": 0.91366,
    "GBP": 0.78643,
    "JPY": 145.88
  }
}
```

Поле `rates` - словарь `{код_валюты: курс_к_базовой}`. Например `"EUR": 0.91366` означает «1 USD = 0.91366 EUR».

In [ ]:
RATES_URL = "https://api.frankfurter.dev/v1/2024-01-15?from=USD"

## 1. `fetch_rates(url)`

**Задача:** загрузить курсы валют по HTTP и вернуть распарсенный JSON.

**Зачем:** изолируем работу с сетью в одной функции - её легко замокать в тестах, легко перенаправить на другой источник.

**Что делает:**
1. Делает GET-запрос через `requests.get(url, timeout=10)`.
2. Вызывает `resp.raise_for_status()` - если сервер вернул 4xx/5xx, эта строка бросит `HTTPError` (мы не глотаем ошибки молча).
3. Возвращает распарсенный JSON (`dict`).

**Подсказка:** `requests.get(...).json()` уже делает парсинг.

In [ ]:
import requests

def fetch_rates(url):
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


## 2. Класс `Currency`

**Задача:** написать класс-обёртку для одной валюты с правильным `__str__`, `__eq__` и `__hash__`.

**Зачем:** вместо словаря или кортежа заворачиваем валюту в типизированный объект - можно сравнивать, печатать осмысленно, класть в `set`.

**Контракт:**

- `__init__(self, code, rate, base="USD")` - сохранить три атрибута: `self.code` (str), `self.rate` (float - приведите через `float(rate)`), `self.base` (str).
- `__str__(self)` - вернуть строку **точного** формата `"{code}: {rate:.4f} per {base}"`. Пример: для `Currency("EUR", 0.91366)` результат `"EUR: 0.9137 per USD"`. Обратите внимание на формат `:.4f` - округление до 4 знаков с обязательными нулями.
- `__eq__(self, other)` - две валюты считаем равными, если у них совпадает `code` (и только код - курс игнорируем, см. Q7).
- `__hash__(self)` - должен быть согласован с `__eq__`. Если два объекта равны через `__eq__`, у них **обязан** быть одинаковый хеш. Простейший способ - `return hash(self.code)`.

**Подсказка:** для `__eq__` сначала проверьте, что `other` - тоже `Currency` (через `isinstance`), иначе возвращайте `NotImplemented` - это правильный способ сообщить Python «я не умею сравниваться с этим типом».

In [ ]:
class Currency:
    def __init__(self, code, rate, base="USD"):
        self.code = str(code)
        self.rate = float(rate)
        self.base = str(base)

    def __str__(self):
        return f"{self.code}: {self.rate:.4f} per {self.base}"

    def __eq__(self, other):
        if not isinstance(other, Currency):
            return NotImplemented
        return self.code == other.code

    def __hash__(self):
        return hash(self.code)


## 3. Класс `RateBook`

**Задача:** собрать все валюты в одну коллекцию-индекс, которая ведёт себя как словарь (`book["EUR"]`, `"EUR" in book`, `len(book)`, `for c in book`).

**Зачем:** даёт O(1) поиск валюты по коду, без линейного перебора списка.

**Контракт:**

- `__init__(self, currencies, base)` - принять iterable объектов `Currency` и строку `base`. Внутри постройте словарь `self._by_code = {c.code: c for c in currencies}` для быстрого доступа. Сохраните также `self.base`.
- `@classmethod from_json(cls, raw_data)` - **classmethod**: на вход распарсенный ответ API (с ключами `"base"` и `"rates"`). Для каждой пары `(code, rate)` в `raw_data["rates"]` создайте `Currency(code, rate, base=raw_data["base"])`. Верните новый экземпляр `cls(...)`.
- `__getitem__(self, code)` - `book["EUR"]` должен возвращать `Currency`. Если кода нет - `KeyError` (стандартное поведение словаря).
- `__len__(self)` - количество валют.
- `__contains__(self, code)` - возвращает `bool`, поддерживает `"EUR" in book`.
- `__iter__(self)` - итерация выдаёт **объекты Currency**, а не коды. То есть `for c in book: print(c.code)` должно работать.

**Подсказка:** все методы реализуются буквально одной строкой делегации к `self._by_code`. Не пишите свой словарь - используйте встроенный.

In [ ]:
class RateBook:
    def __init__(self, currencies, base):
        self._by_code = {c.code: c for c in currencies}
        self.base = base

    @classmethod
    def from_json(cls, raw_data):
        base = raw_data["base"]
        currencies = [
            Currency(code, rate, base=base)
            for code, rate in raw_data["rates"].items()
        ]
        return cls(currencies, base)

    def __getitem__(self, code):
        return self._by_code[code]

    def __len__(self):
        return len(self._by_code)

    def __contains__(self, code):
        return code in self._by_code

    def __iter__(self):
        return iter(self._by_code.values())


## 4. Класс `Converter`

**Задача:** написать класс с двумя методами конвертации - прямой (из базовой валюты в любую другую) и кросс-курс (между двумя любыми валютами через USD).

**Зачем:** бизнес-логика конвертации - один класс отвечает за все формулы, никто больше не считает курсы руками.

Базовая валюта в нашем фиде - USD; все курсы хранятся относительно неё. Поэтому конвертацию между двумя не-базовыми валютами делаем **через** USD (это называется «кросс-курс»).

**Контракт:**

- `__init__(self, book)` - сохранить ссылку на `RateBook` в `self.book`.
- `from_base(self, amount, code)` - сколько валюты `code` получится из `amount` базовой (USD). Формула: `amount * book[code].rate`. Пример: `from_base(1000, "EUR")` -> `1000 * 0.91366 = 913.66`.
- `cross_rate(self, amount, from_code, to_code)` - сколько `to_code` получится из `amount` `from_code`. Считается в два шага: (1) переводим в USD: `amount / book[from_code].rate`, (2) из USD в целевую: `... * book[to_code].rate`. Итого: `amount / book[from_code].rate * book[to_code].rate`.

**Подсказка:** оба метода - одна строка с использованием `self.book[code].rate`.

In [ ]:
class Converter:
    def __init__(self, book):
        self.book = book

    def from_base(self, amount, code):
        return amount * self.book[code].rate

    def cross_rate(self, amount, from_code, to_code):
        return (amount / self.book[from_code].rate) * self.book[to_code].rate


## 5. Класс `CachedRateBook(RateBook)`

**Задача:** унаследовать `RateBook` и добавить лог обращений - чтобы было видно к каким валютам приложение лезет чаще всего.

**Зачем:** в проде хотим видеть, к каким валютам приложение обращается чаще всего - это вход для решений «какие курсы кешировать, какие тарифы оптимизировать».

**Что добавить:**

- В `__init__`: сначала вызовите `super().__init__(currencies, base)`, затем заведите `self._access_log = []`.
- Переопределите `__getitem__(self, code)`: добавьте `code` в `self._access_log` (записываем факт обращения) и верните результат `super().__getitem__(code)` - не дублируйте логику родителя.

**Подсказка:** ключевая идея - **не переписывать** родительскую логику, а только обернуть её. `super()` - ваш друг.

In [ ]:
class CachedRateBook(RateBook):
    def __init__(self, currencies, base):
        super().__init__(currencies, base)
        self._access_log = []

    def __getitem__(self, code):
        self._access_log.append(code)
        return super().__getitem__(code)


## 6. `TurboBook` (diamond MRO)

**Задача:** определить MRO для класса с двумя родителями и впишите его в Q6.

Здесь **писать ничего не нужно** - класс уже объявлен ниже.

**Зачем:** MRO - это порядок, в котором Python ищет методы у класса с несколькими родителями. Здесь у нас diamond-наследование: `TurboBook` наследует от `CachedRateBook` (который от `RateBook`) и от `ObservableMixin` (миксин). Python разрешает это через C3-линеаризацию; результат можно посмотреть через `Cls.__mro__`.

In [ ]:
class ObservableMixin:
    def notify(self, event):
        print("event:", event)


class TurboBook(CachedRateBook, ObservableMixin):
    pass

## Запуск

Прогоните ячейку после того как заполнены все классы и функция. Каждое из 6 значений `Q1..Q6` перенесите в соответствующий инпут на странице задания. Если получаете ошибку - значит что-то не реализовано или реализовано неверно, поправьте ячейку выше и запустите снова.

In [ ]:
data = fetch_rates(RATES_URL)
book = RateBook.from_json(data)
converter = Converter(book)

print("Q1:", book["EUR"].rate)
print("Q2:", round(converter.from_base(1000, "EUR"), 2))
print("Q3:", round(converter.cross_rate(1000, "EUR", "GBP"), 2))
print("Q4:", len(book))
print("Q5:", str(book["EUR"]))
print("Q6:", ", ".join(c.__name__ for c in TurboBook.__mro__))

## 7. Открытый вопрос (manual)

**Задача:** объяснить почему `Currency.__eq__` сравнивает только `code`, а не `rate`.

В Currency мы написали `__eq__` так, что сравнивается **только** `code`, а `rate` игнорируется. Объясните почему это правильно именно для нашего сценария:

1. Что произошло бы, если бы `__eq__` сравнивал и курс тоже? Подумайте про `set`, `dict`-индекс по валюте, обновление курса.
2. Какой инвариант мы хотим сохранить: «один код = одна валюта в нашем приложении»? Или другой?

Ответ 2-3 предложениями - в инпут Q7 на странице задания, его проверяет преподаватель.